## NLP Assignment 3 — Chatbot using Hugging Face Transformers

## Step 1: Install Required Libraries

In [ ]:
# Install required packages
# Run this cell if you're on Google Colab or a fresh environment
!pip install transformers torch --quiet

##  Step 2: Import Libraries

In [ ]:
# Import necessary libraries
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

print("✅ Libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")

## Step 3: Load Pre-trained Model and Tokenizer


In [ ]:
# Define the pre-trained model name from Hugging Face Hub
MODEL_NAME = "microsoft/DialoGPT-medium"

print(f"⏳ Loading tokenizer from '{MODEL_NAME}'...")
# Load the tokenizer for the model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f"⏳ Loading model from '{MODEL_NAME}'... (this may take a moment)")
# Load the pre-trained DialoGPT model
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

# Set model to evaluation mode (disables dropout layers — not training)
model.eval()

print("✅ Model and Tokenizer loaded successfully!")

## Step 4: Define the Response Generation Function

In [ ]:
def generate_response(user_input, chat_history_ids=None):
    # Encode the user input and append EOS token to mark end of this turn
    # EOS (End-of-Sequence) token acts as a separator between conversation turns
    new_user_input_ids = tokenizer.encode(
        user_input + tokenizer.eos_token,
        return_tensors='pt'  # Return as PyTorch tensor
    )

    if chat_history_ids is not None:
        bot_input_ids = torch.cat([chat_history_ids, new_user_input_ids], dim=-1)
    else:
        # First turn: no history yet
        bot_input_ids = new_user_input_ids

    with torch.no_grad():
        chat_history_ids = model.generate(
            bot_input_ids,
            max_length=1000,                          # Max total token length
            pad_token_id=tokenizer.eos_token_id,      # Pad with EOS token
            do_sample=True,                           # Use sampling (not greedy)
            top_p=0.92,                               # Nucleus sampling
            temperature=0.75,                         # Response creativity level
            repetition_penalty=1.3                    # Penalize repeated phrases
        )

    response = tokenizer.decode(
        chat_history_ids[:, bot_input_ids.shape[-1]:][0],
        skip_special_tokens=True  # Remove special tokens like EOS from output
    )

    return response, chat_history_ids


print("✅ Response generation function defined.")

## Step 5: Run the Chatbot

In [ ]:
def run_chatbot():
    """
    Main function to run the interactive console-based chatbot.
    Maintains conversation history across multiple turns.
    Exits when the user types 'exit' or 'quit'.
    """
    print("Chatbot: Hello! I am your AI assistant. How can I help you today?")
    print("(Type 'exit' or 'quit' to end the conversation)")


    # Initialize conversation history as None (empty at the start)
    chat_history_ids = None

    # Continuous conversation loop
    while True:
        # Prompt the user for input
        user_input = input("\nYou: ").strip()

        # Check for empty input — ask again politely
        if not user_input:
            print("Chatbot: It seems you didn't type anything. Please go ahead!")
            continue

        # Exit condition: user types 'exit' or 'quit' (case-insensitive)
        if user_input.lower() in ["exit", "quit"]:
            print("\nChatbot: It was great talking to you! Goodbye! 👋")
            print("=" * 60)
            break

        # Generate a response using the model, passing conversation history
        response, chat_history_ids = generate_response(user_input, chat_history_ids)

        # Display the chatbot's response
        print(f"\nChatbot: {response}")


# ▶️ Start the chatbot
run_chatbot()

## Step 6: Simulated Conversation Demo

In [ ]:
def demo_conversation(test_inputs):
    """
    Simulate a conversation with predefined inputs for demonstration.

    Args:
        test_inputs (list of str): A list of user messages to simulate.
    """
    print("=" * 60)
    print("       DEMO: Simulated Chatbot Conversation")
    print("=" * 60)
    print("Chatbot: Hello! I am your AI assistant. How can I help you today?")
    print("-" * 60)

    # No history at the beginning
    chat_history_ids = None

    for user_input in test_inputs:
        print(f"\nYou: {user_input}")

        # Generate and display the response
        response, chat_history_ids = generate_response(user_input, chat_history_ids)
        print(f"Chatbot: {response}")

    print("\n" + "-" * 60)
    print("[Demo conversation ended]")
    print("=" * 60)


# Predefined test messages to simulate a conversation
sample_inputs = [
    "Hello",
    "What is Artificial Intelligence?",
    "Who created Python?",
    "What is machine learning?",
    "Thank you"
]

# Run the demo
demo_conversation(sample_inputs)